# Face Attendance System in Python (Jupyter Notebook)
The codebase in this folder is a **React Web Application**, which is not designed to run in a Jupyter Notebook. 

However, below is the **Python equivalent** of the face attendance system that you can run right here in Jupyter. It uses `OpenCV` and `face_recognition` to track faces through your webcam.

### Prerequisites:
Run `pip install opencv-python face_recognition numpy` in your environment before running the face tracker.

In [1]:
import cv2
import json
import datetime
import numpy as np
import face_recognition

# 1. Load known face encodings and names
# Uncomment the lines below and point them to your own image to actually recognize yourself!
# known_image = face_recognition.load_image_file('your_photo.jpg')
# known_encoding = face_recognition.face_encodings(known_image)[0]
known_face_encodings = []
known_face_names = [] # E.g., ["Ronak"]

# To keep track of today's attendance so we don't log the same person twice
attendance_log = []

# 2. Start Webcam
video_capture = cv2.VideoCapture(0)
print("Started face tracking. To exit, click the webcam window and press 'q'.")

while True:
    ret, frame = video_capture.read()
    if not ret:
        print("Failed to grab frame. Is your webcam being used by another app?")
        break

    # Resize frame to 1/4 size for faster processing
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    # Convert BGR (OpenCV) to RGB (face_recognition)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
    
    # Find all face locations and encodings
    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    face_names = []
    for face_encoding in face_encodings:
        name = "Unknown"
        
        if len(known_face_encodings) > 0:
            # See if face is a match for known face
            matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
            face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
            best_match_index = np.argmin(face_distances)
            if matches[best_match_index]:
                name = known_face_names[best_match_index]
        else:
            # If no known faces provided, just alert that a face is detected
            name = "Face Detected"
            
        face_names.append(name)

        # 3. Log Attendance
        if name != "Unknown":
            now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            if not any(record['name'] == name for record in attendance_log):
                attendance_log.append({"name": name, "time": now})
                print(f"[{now}] Attendance recorded for {name}")

    # 4. Display Results on Screen
    for (top, right, bottom, left), name in zip(face_locations, face_names):
        # Scale back up since we scaled to 1/4 layout
        top *= 4
        right *= 4
        bottom *= 4
        left *= 4

        # Draw box around face
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        # Draw label beneath face
        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), (0, 255, 0), cv2.FILLED)
        cv2.putText(frame, name, (left + 6, bottom - 6), cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)

    cv2.imshow('Face Attendance System', frame)

    # Check if 'q' gets pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video_capture.release()
cv2.destroyAllWindows()

# 5. Save attendance record log to disk
if attendance_log:
    with open("attendance_jupyter_log.json", "w") as f:
        json.dump(attendance_log, f, indent=4)
    print("\nAttendance saved to attendance_jupyter_log.json!")
else:
    print("\nNo attendance to log.")


ModuleNotFoundError: No module named 'cv2'